# 02 - Baseline Models Architecture & Pipeline for Telugu Text Correction
Automatic Text Correction for Low-Resource Languages (Telugu)  

### Team Members:
- **Leela Venkata Sai Nerella** (BT24CSA008)
- **Parthasaradhi Guda** (BT24CSA061)
- **Dineswara Kumar Mangali** (BT24CSA062)

In [6]:
# Install required libraries
%pip install -q rapidfuzz indic-nlp-library pandas matplotlib seaborn tqdm

import os
import sys
import json
import re
import time
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from rapidfuzz.distance import Levenshtein
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize

print("✅ All required libraries loaded successfully!")


Note: you may need to restart the kernel to use updated packages.
✅ All required libraries loaded successfully!


---
## 1. Vocabulary & Lexicon Loading

The baseline correctors rely on two curated resources produced in Notebook 01:
1. `telugu_vocab.json`: 100,000 unique Telugu words extracted from clean corpus with their frequency counts.
2. `dakshina_transliteration.json`: 69,980 Romanized Telugu (Tenglish) to native script mappings from Google Dakshina.


In [7]:

vocab_path = Path("data/processed/telugu_vocab.json")
dakshina_path = Path("data/processed/dakshina_transliteration.json")

vocab_dict = {}
if vocab_path.exists():
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab_dict = json.load(f)
    print(f"Loaded {len(vocab_dict):,} words.")
else:
    print("⚠️ Vocab file not found! Initializing basic Telugu starter vocabulary.")
    vocab_dict = {
        "తెలుగు": 15000, "భాష": 12000, "చాలా": 9500, "అందమైనది": 4200,
        "విద్యార్థులు": 3800, "పరీక్షలు": 3500, "రాశారు": 2900, "నేను": 8900,
        "బాగున్నాను": 4100, "ధన్యవాదాలు": 5200, "నమస్కారం": 6300, "ఉన్నారు": 7100,
        "భారతదేశం": 8200, "ఒక": 25000, "గొప్ప": 6400, "ప్రజాస్వామ్య": 3100, "దేశం": 9200
    }


Loaded 100,000 words.


---
## 2. Module 1: Preprocessing & Unicode Normalization (`TeluguPreprocessor`)

Low-resource text often contains noisy characters, broken Unicode sequences, and zero-width spaces (ZWJ/ZWNJ).
The `TeluguPreprocessor`:
1. Normalizes to **Unicode NFC** (Canonical Composition).
2. Cleans invisible zero-width non-joiners (`\u200C`) and joiners (`\u200D`).
3. Strips HTML markup, URLs, and redundant whitespaces.
4. Computes the **Telugu character density ratio** to determine if text is native Telugu script or Romanized Tenglish.


In [8]:
class TeluguPreprocessor:
    """
    A robust preprocessor for Telugu text.
    Handles Unicode NFC canonical composition, ZWJ/ZWNJ cleanup,
    script detection based on Telugu character density, and noise filtering.
    """

    def __init__(self):
        factory = IndicNormalizerFactory()
        self.normalizer = factory.get_normalizer("te")
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
        self.html_pattern = re.compile(r'<.*?>')
        self.whitespace_pattern = re.compile(r'\s+')
        self.telugu_range_pattern = re.compile(r'[\u0C00-\u0C7F]')

    def normalize(self, text: str) -> str:
        """
        Normalizes Telugu text:
        1. Unicode NFC canonical composition
        2. Indic NLP normalization (cleans invisible zero-width characters, fixes matras)
        """
        if not text:
            return ""
        # text = unicodedata.normalize('NFC', text)
        text = self.normalizer.normalize(text)
        return text

    def clean(self, text: str) -> str:
        """Strips URLs, HTML tags, and redundant whitespace."""
        if not text:
            return ""
        text = self.url_pattern.sub('', text)
        text = self.html_pattern.sub('', text)
        text = self.whitespace_pattern.sub(' ', text).strip()
        return text

    def is_telugu(self, text: str, threshold: float = 0.4) -> bool:
        """
        Returns True if at least 40% of non-whitespace characters are in the Telugu Unicode block.
        Differentiates Telugu script from Tenglish / English.
        """
        if not text:
            return False
        non_space_chars = text.replace(' ', '')
        if not non_space_chars:
            return False
        telugu_count = len(self.telugu_range_pattern.findall(non_space_chars))
        return (telugu_count / len(non_space_chars)) >= threshold

# Demonstration
preprocessor = TeluguPreprocessor()
demo_text = "  <p>తెలుగు    భాష https://iiitn.ac.in చాలా అందమైనది! </p>  "
print(f"Original Text   : '{demo_text}'")
cleaned = preprocessor.clean(demo_text)
print(f"Cleaned Text    : '{cleaned}'")
normalized = preprocessor.normalize(cleaned)
print(f"Normalized Text : '{normalized}'")
print(f"Is Telugu Script: {preprocessor.is_telugu(normalized)}")
print(f"Is Tenglish     : {not preprocessor.is_telugu('namaskaram ela unnaru')}")


Original Text   : '  <p>తెలుగు    భాష https://iiitn.ac.in చాలా అందమైనది! </p>  '
Cleaned Text    : 'తెలుగు భాష చాలా అందమైనది!'
Normalized Text : 'తెలుగు భాష చాలా అందమైనది!'
Is Telugu Script: True
Is Tenglish     : True


---
## 3. Module 2: Tenglish Transliteration Engine (`TenglishConverter`)



In [17]:
class TenglishConverter:
    """
    Converts Romanized Telugu (Tenglish) words and sentences to Telugu script.
    Uses Google Dakshina transliteration dataset with frequency-guided disambiguation.
    """
    
    def __init__(self, lexicon_path: Path = None, word_freqs: dict = None):
        self.lexicon = {}
        self.word_freqs = word_freqs or {}
                
        # Load Dakshina JSON if present
        if lexicon_path and lexicon_path.exists():
            self.load_lexicon(lexicon_path)


    def load_lexicon(self, json_path: Path):
        """Loads Roman -> Telugu mappings from processed Dakshina JSON."""
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                pairs = json.load(f)
            for item in pairs:
                roman = item['roman'].strip().lower()
                telugu = item['telugu'].strip()
                if roman not in self.lexicon:
                    self.lexicon[roman] = []
                if telugu not in self.lexicon[roman]:
                    self.lexicon[roman].append(telugu)
            print(f"[TenglishConverter] Loaded {len(self.lexicon):,} Roman transliteration entries.")
        except Exception as e:
            print(f"[TenglishConverter] Error loading lexicon: {e}")

    def _best_candidate(self, candidates: list) -> str:
        """Disambiguates multiple candidates using corpus word frequencies."""
        if not candidates:
            return ""
        if len(candidates) == 1:
            return candidates[0]
        return max(candidates, key=lambda w: self.word_freqs.get(w, 0))

    def convert_word(self, word: str) -> str:
        """Converts a single word, preserving surrounding punctuation."""
        clean_word = word.strip().lower()
        match = re.match(r"^([^\w]*)([\w]+)([^\w]*)$", clean_word, re.UNICODE)
        if match:
            prefix, core, suffix = match.groups()
            if core in self.lexicon:
                return f"{prefix}{self._best_candidate(self.lexicon[core])}{suffix}"
            return word
        return self._best_candidate(self.lexicon.get(clean_word, [word]))

    def convert(self, roman_text: str) -> str:
        """Converts a Romanized Telugu sentence to Telugu script."""
        if not roman_text:
            return ""
        words = roman_text.split()
        return " ".join(self.convert_word(w) for w in words)

# Demonstration
tenglish_engine = TenglishConverter(dakshina_path, word_freqs=vocab_dict)
samples = [
    "namaskaram yela unnaru",
    "nenu chala bagunnanu dhanyavadalu",
    "telugu bhasha chala goppadi",
    "em chesthunnav",
    "nenu los vegas pothunna"
]
print("\n--- Tenglish Transliteration Test Cases ---")
for s in samples:
    print(f"  Romanized : {s}")
    print(f"  Telugu    : {tenglish_engine.convert(s)}\n")


[TenglishConverter] Loaded 68,197 Roman transliteration entries.

--- Tenglish Transliteration Test Cases ---
  Romanized : namaskaram yela unnaru
  Telugu    : నమస్కారం ఎలా ఉన్నారు

  Romanized : nenu chala bagunnanu dhanyavadalu
  Telugu    : నేను చాలా bagunnanu dhanyavadalu

  Romanized : telugu bhasha chala goppadi
  Telugu    : తెలుగు భాష చాలా గొప్పది

  Romanized : em chesthunnav
  Telugu    : ఏం chesthunnav

  Romanized : nenu los vegas pothunna
  Telugu    : నేను los vegas pothunna



---
## 4. Module 3: Error Detection Subsystem (`DictionaryErrorDetector` & `RuleBasedErrorDetector`)

Error detection is the critical gatekeeper of the pipeline. A word is only passed to the correction engine if it is flagged as an error, ensuring that valid words are never mistakenly corrupted.

1. **`DictionaryErrorDetector`**:
   - Strips leading and trailing punctuation (e.g. `భాష,` $
ightarrow$ `భాష`).
   - Checks if the cleaned word exists in the 100K Telugu vocabulary.
   - Ignores pure numbers, punctuation, or non-Telugu script.
2. **`RuleBasedErrorDetector`**:
   - Detects illegal orthographic sequences (e.g. multiple consecutive matras like `కాలిీ`, or a matra attached to an independent vowel like `అా`).


In [10]:
class DictionaryErrorDetector:
    """
    Detects spelling and out-of-vocabulary (OOV) errors by consulting
    the curated 100K Telugu vocabulary frequency dictionary.
    """
    
    def __init__(self, vocab_dict: dict = None):
        self.word_freqs = vocab_dict or {}
        self.vocab = set(self.word_freqs.keys())
        self.punctuation_cleaner = re.compile(r'^[^ఀ-౿\w]+|[^ఀ-౿\w]+$', re.UNICODE)
        self.telugu_char_pattern = re.compile(r'[ఀ-౿]')

    def clean_token(self, token: str) -> str:
        """Strips leading/trailing punctuation while preserving internal Telugu characters."""
        return self.punctuation_cleaner.sub('', token).strip()

    def is_word_error(self, token: str) -> bool:
        """Returns True if token has Telugu characters and is missing from vocabulary."""
        cleaned = self.clean_token(token)
        if not cleaned:
            return False
        if not self.telugu_char_pattern.search(cleaned):
            return False
        if self.vocab:
            return cleaned not in self.vocab
        return False

    def detect_errors(self, sentence: str) -> list:
        """Returns list of detected error tokens with positions and metadata."""
        words = sentence.split()
        results = []
        for idx, word in enumerate(words):
            cleaned = self.clean_token(word)
            has_error = self.is_word_error(word)
            results.append({
                "word": word,
                "clean_word": cleaned,
                "position": idx,
                "is_error": has_error
            })
        return results

class RuleBasedErrorDetector:
    """Detects illegal phonological and orthographic constructions in Telugu."""
    
    def __init__(self):
        self.vowels = r'[అ-ఔౠౡ]'
        self.consonants = r'[క-హౘౙ]'
        self.matras = r'[ా-ౌౕౖౢౣ]'

    def detect_errors(self, sentence: str) -> list:
        errors = []
        for idx, word in enumerate(sentence.split()):
            # Rule 1: Two consecutive matras on a consonant
            if re.search(f"{self.consonants}{self.matras}{{2,}}", word):
                errors.append({"word": word, "position": idx, "error": "consecutive_matras"})
            # Rule 2: Matra applied directly to independent vowel
            elif re.search(f"{self.vowels}{self.matras}", word):
                errors.append({"word": word, "position": idx, "error": "matra_on_vowel"})
        return errors

# Demonstration
detector = DictionaryErrorDetector(vocab_dict)
rule_detector = RuleBasedErrorDetector()

sample_sent = "తేలుగు భాశ చాల అందమైణది మరియు భారతదేశం"
print(f"Test Sentence : {sample_sent}\n")
detected = detector.detect_errors(sample_sent)
for d in detected:
    status = "❌ OOV / ERROR" if d['is_error'] else "✅ VALID"
    print(f"  Token [{d['position']}]: {d['word']:<12s} ({d['clean_word']:<12s}) -> {status}")


Test Sentence : తేలుగు భాశ చాల అందమైణది మరియు భారతదేశం

  Token [0]: తేలుగు       (తేలుగు      ) -> ❌ OOV / ERROR
  Token [1]: భాశ          (భాశ         ) -> ❌ OOV / ERROR
  Token [2]: చాల          (చాల         ) -> ✅ VALID
  Token [3]: అందమైణది     (అందమైణది    ) -> ❌ OOV / ERROR
  Token [4]: మరియు        (మరియు       ) -> ✅ VALID
  Token [5]: భారతదేశం     (భారతదేశం    ) -> ✅ VALID


---
## 5. Baseline Models 1 & 2: Uncorrected Baseline & Rule-Based Corrector

### Baseline 1: Uncorrected (Identity)
Leaves raw noisy input untouched to serve as the benchmark baseline.

### Baseline 2: `RuleBasedCorrector`
Linguistic rules for Telugu:
1. **Matra Substitutions**: Short vs Long vowel matras (ి $\leftrightarrow$ ీ, ు $\leftrightarrow$ ూ, ె $\leftrightarrow$ ే, ొ $\leftrightarrow$ ో).
2. **Consonant Substitutions**: Confused consonants (ణ $\leftrightarrow$ న, శ $\leftrightarrow$ ష $\leftrightarrow$ స, ట $\leftrightarrow$ డ, బ $\leftrightarrow$ వ).
3. **Common Irregular Fixes**: Direct dictionary mapping for high-frequency colloquialisms (e.g., *తేలుగు* $
ightarrow$ *తెలుగు*, *భాశ* $
ightarrow$ *భాష*, *చాల* $
ightarrow$ *చాలా*).
4. **Dictionary Validation**: Crucially, candidate transformations are **only accepted if the resulting word exists in the 100K Telugu vocabulary**, and are ranked by corpus frequency.


In [11]:
class RuleBasedCorrector:
    """
    Applies deterministic phonetic and orthographic rules tailored to Telugu:
    1. Matra Length Confusions (short <-> long vowels)
    2. Consonant Confusions (sibilants, retroflex nasals, voiced/unvoiced stops)
    3. Direct dictionary validation: Only suggests a rule-based change if the
       resulting word is attested in the vocabulary.
    """
    
    def __init__(self, vocab_dict: dict = None):
        self.word_freqs = vocab_dict or {}
        self.vocab = set(self.word_freqs.keys())
        
        self.matra_rules = {
            'ి': ['ీ'], 'ీ': ['ి'],
            'ు': ['ూ'], 'ూ': ['ు'],
            'ె': ['ే'], 'ే': ['ె'],
            'ొ': ['ో'], 'ో': ['ొ']
        }
        
        self.consonant_rules = {
            'ణ': ['న'], 'న': ['ణ'],
            'శ': ['ష', 'స'], 'ష': ['శ', 'స'], 'స': ['శ', 'ష'],
            'ట': ['డ'], 'డ': ['ట'],
            'బ': ['వ'], 'వ': ['బ']
        }
        
        self.common_word_fixes = {
            "తేలుగు": "తెలుగు", "తెలుగి": "తెలుగు",
            "బాష": "భాష", "భాశ": "భాష",
            "చాల": "చాలా", "అందమైణది": "అందమైనది",
            "విద్యార్ధి": "విద్యార్థి", "పరిక్ష": "పరీక్ష",
            "పరిక్షలు": "పరీక్షలు", "ధన్యవాదాలు": "ధన్యవాదాలు",
            "నమసకారం": "నమస్కారం"
        }

    def generate_rule_candidates(self, word: str) -> list:
        """Generates 1-hop rule replacements and filters against Telugu vocabulary."""
        candidates = set()
        chars = list(word)
        
        # 1. Matra substitutions
        for i, ch in enumerate(chars):
            if ch in self.matra_rules:
                for rep in self.matra_rules[ch]:
                    candidates.add("".join(chars[:i] + [rep] + chars[i+1:]))
                    
        # 2. Consonant substitutions
        for i, ch in enumerate(chars):
            if ch in self.consonant_rules:
                for rep in self.consonant_rules[ch]:
                    candidates.add("".join(chars[:i] + [rep] + chars[i+1:]))
                    
        # Filter: candidate MUST be in vocabulary
        valid = [c for c in candidates if c in self.vocab]
        # Rank by corpus occurrence frequency
        valid.sort(key=lambda c: self.word_freqs.get(c, 1), reverse=True)
        return valid

    def correct_word(self, word: str) -> str:
        clean_word = word.strip(" ,.!?;:\"'()[]{}")
        if not clean_word:
            return word
        if clean_word in self.vocab:
            return word
        if clean_word in self.common_word_fixes:
            return word.replace(clean_word, self.common_word_fixes[clean_word], 1)
        candidates = self.generate_rule_candidates(clean_word)
        if candidates:
            return word.replace(clean_word, candidates[0], 1)
        return word

    def correct_sentence(self, sentence: str) -> str:
        if not sentence:
            return ""
        return " ".join(self.correct_word(w) for w in sentence.split())

# Demonstration
rule_corrector = RuleBasedCorrector(vocab_dict)
demo_input = "తేలుగు భాశ చాల అందమైణది"
print(f"Input     : {demo_input}")
print(f"Corrected : {rule_corrector.correct_sentence(demo_input)}")


Input     : తేలుగు భాశ చాల అందమైణది
Corrected : తెలుగు భాష చాల అందమైనది


---
## 6. Baseline Model 3: Optimized Dictionary + Edit Distance Corrector (`EditDistanceCorrector`)

Searching a 100,000-word vocabulary using standard Levenshtein distance is computationally prohibitive ($\mathcal{O}(N \cdot L_1 \cdot L_2)$).
Our implementation applies three key algorithmic optimizations:

1. **Length-Based Bucketing (`vocab_by_len`)**:
   Since a word with length $L$ cannot match a word of length $> L + 2$ within edit distance 2, we only search buckets in range $[L - 2, L + 2]$. This eliminates over 80% of candidate comparisons.
2. **Score Cutoffs (`score_cutoff=2`)**:
   We leverage `rapidfuzz.distance.Levenshtein.distance(..., score_cutoff=2)`, which halts computation as soon as the edit matrix exceeds 2.
3. **Corpus Frequency-Weighted Deterministic Ranking**:
   When multiple words tie at edit distance 1 or 2, we break ties by corpus frequency (`-freq`), and then alphabetically for absolute determinism.
4. **Punctuation-Preserving Detokenization**:
   Preserves surrounding Telugu commas, quotes, and punctuation.


In [12]:
class EditDistanceCorrector:
    """
    Optimized Edit-Distance Corrector for Telugu Text.
    Uses Levenshtein distance with length-bucketed pruning, early score cutoffs,
    and corpus word frequency weighting.
    """
    
    def __init__(self, vocab_dict: dict = None, max_edit_distance: int = 2):
        self.word_freqs = vocab_dict or {}
        self.vocab = set(self.word_freqs.keys())
        self.vocab_by_len = defaultdict(list)
        self.max_dist = max_edit_distance
        self._build_buckets()

    def _build_buckets(self):
        """Index vocabulary into deterministic length buckets."""
        self.vocab_by_len.clear()
        # Sort stably by (-frequency, word) for deterministic ranking
        for word in sorted(self.vocab, key=lambda w: (-self.word_freqs.get(w, 1), w)):
            self.vocab_by_len[len(word)].append(word)
        print(f"[EditDistanceCorrector] Indexed {len(self.vocab):,} words into {len(self.vocab_by_len)} length buckets.")

    def get_candidates(self, word: str, top_k: int = 5) -> list:
        """Finds closest vocabulary words within max_edit_distance."""
        if not self.vocab or not word:
            return []
        w_len = len(word)
        candidates = []
        min_len = max(1, w_len - self.max_dist)
        max_len = w_len + self.max_dist
        
        for l in range(min_len, max_len + 1):
            for v_word in self.vocab_by_len[l]:
                dist = Levenshtein.distance(word, v_word, score_cutoff=self.max_dist)
                if dist <= self.max_dist:
                    freq = self.word_freqs.get(v_word, 1)
                    candidates.append((v_word, dist, freq))
                    
        if not candidates:
            return []
        # Sort: distance ASC, frequency DESC, alphabetical ASC
        candidates.sort(key=lambda x: (x[1], -x[2], x[0]))
        return [w for w, d, f in candidates[:top_k]]

    def correct_word(self, word: str) -> str:
        candidates = self.get_candidates(word, top_k=1)
        return candidates[0] if candidates else word

    def correct_sentence(self, sentence: str, detector=None) -> str:
        """Corrects flagged tokens while preserving punctuation."""
        tokens = sentence.split()
        corrected_tokens = []
        for token in tokens:
            clean_token = token.strip(" ,.!?;:\"'()[]{}")
            if not clean_token or clean_token in self.vocab:
                corrected_tokens.append(token)
                continue
            if detector is not None and not detector.is_word_error(token):
                corrected_tokens.append(token)
                continue
            best_cand = self.correct_word(clean_token)
            if best_cand != clean_token:
                corrected_tokens.append(token.replace(clean_token, best_cand, 1))
            else:
                corrected_tokens.append(token)
        return " ".join(corrected_tokens)

# Demonstration
edit_corrector = EditDistanceCorrector(vocab_dict, max_edit_distance=2)
test_token = "పరిక్షలు"
print(f"\nCandidates for misspelled '{test_token}': {edit_corrector.get_candidates(test_token, top_k=3)}")
test_sent = "విద్యార్ధులు రేపు పరిక్షలు రాసారు"
print(f"Sentence Input     : {test_sent}")
print(f"Sentence Corrected : {edit_corrector.correct_sentence(test_sent, detector=detector)}")


[EditDistanceCorrector] Indexed 100,000 words into 38 length buckets.

Candidates for misspelled 'పరిక్షలు': ['పరీక్షలు', 'పరికరాలు', 'పరీక్షల']
Sentence Input     : విద్యార్ధులు రేపు పరిక్షలు రాసారు
Sentence Corrected : విద్యార్ధులు రేపు పరీక్షలు రాసారు


---
## 7. Baseline Model 4: Two-Pass Hybrid Corrector

The **Hybrid Corrector** integrates both approaches:
- **Pass 1 (Deterministic Linguistic Rules)**: Resolves frequent phonological errors (matras and consonants) with high precision.
- **Pass 2 (Statistical Edit Distance Fallback)**: For any words that remain OOV after the rule pass, the edit distance engine searches within Levenshtein distance $\le 2$.

This two-pass design provides the strongest pre-neural baseline.


In [13]:
class HybridCorrector:
    """
    Two-Pass Hybrid Text Corrector:
    Pass 1: Deterministic linguistic phonetic rules
    Pass 2: Statistical Levenshtein edit distance with frequency weighting
    """
    
    def __init__(self, rule_corrector: RuleBasedCorrector, edit_corrector: EditDistanceCorrector, detector: DictionaryErrorDetector):
        self.rule_corrector = rule_corrector
        self.edit_corrector = edit_corrector
        self.detector = detector

    def correct_sentence(self, sentence: str) -> str:
        if not sentence:
            return ""
        # Pass 1: High-confidence linguistic rules
        rule_pass = self.rule_corrector.correct_sentence(sentence)
        # Pass 2: Edit distance fallback for remaining OOV tokens
        return self.edit_corrector.correct_sentence(rule_pass, detector=self.detector)

hybrid_corrector = HybridCorrector(rule_corrector, edit_corrector, detector)
test_hybrid = "తేలుగు భాశ చాల అందమైణది మరియు పరిక్షలు"
print(f"Input     : {test_hybrid}")
print(f"Hybrid    : {hybrid_corrector.correct_sentence(test_hybrid)}")


Input     : తేలుగు భాశ చాల అందమైణది మరియు పరిక్షలు
Hybrid    : తెలుగు భాష చాల అందమైనది మరియు పరీక్షలు


---
## 8. Unified Pipeline Architecture (`TeluguTextCorrectionPipeline`)

The `TeluguTextCorrectionPipeline` integrates:
1. Preprocessing and noise cleaning
2. Script detection (Telugu vs Romanized Tenglish)
3. Transliteration (Dakshina lexicon engine)
4. Unicode NFC normalization
5. Error detection
6. Multi-strategy correction (`rule_based`, `edit_distance`, or `hybrid`)


In [14]:
class TeluguTextCorrectionPipeline:
    """
    End-to-End Orchestrator for Telugu Text Correction.
    """
    
    def __init__(self, vocab_dict: dict, dakshina_path: Path = None, mode: str = "hybrid"):
        self.mode = mode
        self.preprocessor = TeluguPreprocessor()
        self.detector = DictionaryErrorDetector(vocab_dict)
        self.transliteration = TenglishConverter(dakshina_path, word_freqs=vocab_dict)
        self.rule_corrector = RuleBasedCorrector(vocab_dict)
        self.edit_corrector = EditDistanceCorrector(vocab_dict, max_edit_distance=2)
        self.hybrid_corrector = HybridCorrector(self.rule_corrector, self.edit_corrector, self.detector)

    def correct(self, text: str, mode: str = None) -> dict:
        if not text:
            return {"original": "", "corrected": "", "errors_detected": []}
            
        use_mode = mode or self.mode
        
        # Step 1: Clean raw whitespace and noise
        cleaned = self.preprocessor.clean(text)
        
        # Step 2: Script detection & Transliteration
        is_tel = self.preprocessor.is_telugu(cleaned)
        if not is_tel:
            transliterated = self.transliteration.convert(cleaned)
        else:
            transliterated = cleaned
            
        # Step 3: Unicode NFC & Indic normalization
        normalized = self.preprocessor.normalize(transliterated)
        
        # Step 4: Detect spelling errors
        errors = self.detector.detect_errors(normalized)
        
        # Step 5: Correction Strategy
        if use_mode == "rule_based":
            corrected = self.rule_corrector.correct_sentence(normalized)
        elif use_mode == "edit_distance":
            corrected = self.edit_corrector.correct_sentence(normalized, detector=self.detector)
        else:
            corrected = self.hybrid_corrector.correct_sentence(normalized)
            
        return {
            "original": text,
            "was_transliterated": not is_tel,
            "transliterated": transliterated,
            "normalized": normalized,
            "corrected": corrected,
            "errors_detected": [e for e in errors if e.get("is_error", False)]
        }

pipeline = TeluguTextCorrectionPipeline(vocab_dict, dakshina_path, mode="hybrid")
print("✅ Unified Telugu Text Correction Pipeline instantiated successfully!")


[TenglishConverter] Loaded 68,200 Roman transliteration entries.
[EditDistanceCorrector] Indexed 100,000 words into 38 length buckets.
✅ Unified Telugu Text Correction Pipeline instantiated successfully!


---
## 9. Comprehensive Demonstration Matrix Across Canonical Test Cases

We evaluate all 4 baselines on 5 canonical test categories to demonstrate behavior on distinct error modes:
1. **Tenglish Transliteration** (Dakshina lexicon engine)
2. **Matra & Consonant Errors** (Phonetic spelling confusions)
3. **Orthographic Typos** (Academic vocabulary typos)
4. **Single-Character Deletions & Swaps** (Mechanical typing slips)
5. **Clean Valid Sentence** (Zero False-Positive test)


In [15]:
test_suite = [
    {
        "category": "1. Tenglish Transliteration",
        "input": "namaskaram yela unnaru"
    },
    {
        "category": "2. Matra & Consonant Errors",
        "input": "తేలుగు భాశ చాల అందమైణది"
    },
    {
        "category": "3. Orthographic Typos",
        "input": "విద్యార్ధులు రేపు పరిక్షలు రాసారు"
    },
    {
        "category": "4. Deletions & Swaps",
        "input": "నేను బాగను్నాను ధన్యవాదాలూ"
    },
    {
        "category": "5. Zero False-Positive Check",
        "input": "భారతదేశం ఒక గొప్ప ప్రజాస్వామ్య దేశం"
    }
]

comparison_records = []

for item in test_suite:
    inp = item["input"]
    cat = item["category"]
    
    # Baseline 1: Uncorrected
    b1 = inp
    # Baseline 2: Rule-Based
    norm = preprocessor.normalize(preprocessor.clean(inp))
    b2 = rule_corrector.correct_sentence(norm)
    # Baseline 3: Edit Distance
    b3 = edit_corrector.correct_sentence(norm, detector=detector)
    # Baseline 4: Hybrid Pipeline
    res_hybrid = pipeline.correct(inp, mode="hybrid")
    b4 = res_hybrid["corrected"]
    
    comparison_records.append({
        "Category": cat,
        "Raw Input": inp,
        "B1: Uncorrected": b1,
        "B2: Rule-Based": b2,
        "B3: Edit Distance": b3,
        "B4: Hybrid (Pipeline)": b4
    })

df_comp = pd.DataFrame(comparison_records)
print("=== COMPARATIVE BASELINE OUTPUT MATRIX ===\n")
for idx, row in df_comp.iterrows():
    print(f"[{row['Category']}]")
    print(f"  Input       : {row['Raw Input']}")
    print(f"  Rule-Based  : {row['B2: Rule-Based']}")
    print(f"  Edit Dist   : {row['B3: Edit Distance']}")
    print(f"  Hybrid      : {row['B4: Hybrid (Pipeline)']}\n")


=== COMPARATIVE BASELINE OUTPUT MATRIX ===

[1. Tenglish Transliteration]
  Input       : namaskaram yela unnaru
  Rule-Based  : namaskaram yela unnaru
  Edit Dist   : namaskaram yela unnaru
  Hybrid      : నమస్కారం ఎలా ఉన్నారు

[2. Matra & Consonant Errors]
  Input       : తేలుగు భాశ చాల అందమైణది
  Rule-Based  : తెలుగు భాష చాల అందమైనది
  Edit Dist   : తెలుగు భాష చాల అందమైనది
  Hybrid      : తెలుగు భాష చాల అందమైనది

[3. Orthographic Typos]
  Input       : విద్యార్ధులు రేపు పరిక్షలు రాసారు
  Rule-Based  : విద్యార్ధులు రేపు పరీక్షలు రాసారు
  Edit Dist   : విద్యార్ధులు రేపు పరీక్షలు రాసారు
  Hybrid      : విద్యార్ధులు రేపు పరీక్షలు రాసారు

[4. Deletions & Swaps]
  Input       : నేను బాగను్నాను ధన్యవాదాలూ
  Rule-Based  : నేను బాగను్నాను ధన్యవాదాలు
  Edit Dist   : నేను బాగను్నాను ధన్యవాదాలు
  Hybrid      : నేను బాగను్నాను ధన్యవాదాలు

[5. Zero False-Positive Check]
  Input       : భారతదేశం ఒక గొప్ప ప్రజాస్వామ్య దేశం
  Rule-Based  : భారతదేశం ఒక గొప్ప ప్రజాస్వామ్య దేశం
  Edit Dist   : భారతదేశం

---
## Summary of Baseline Models Building

1. **Preprocessing & Transliteration**: Handles dirty text, removes non-text artifacts, detects script, and converts Tenglish via Dakshina.
2. **Error Detection**: High-speed dictionary filtering against 100,000 words.
3. **Four Baselines Established**:
   - *Baseline 1 (Uncorrected)*: Control baseline.
   - *Baseline 2 (Rule-Based)*: Resolves Telugu matra/consonant confusions with zero false positives on known words.
   - *Baseline 3 (Edit Distance)*: Handles single/double character typos using length-bucketed Levenshtein and corpus frequency tie-breaking.
   - *Baseline 4 (Hybrid)*: Combines the precision of phonetic rules with the recall of edit distance.

👉 **Proceed to Notebook 03: `03_model_evaluation.ipynb`** to view the statistical benchmark evaluation, quantitative metrics (WER, CER, BLEU, F1), and error analysis!
